# Testing the trained 3D Shapes VAE

Loads the weights saved by `setuo.ipynb` (`vae_trained.pt`) and the 3D Shapes dataset, then re-runs the **Mutual Information Gap (MIG)** disentanglement check against the trained model — without needing to retrain.

In [1]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

PyTorch version: 2.12.1
CUDA available: False
Using device: cpu


In [2]:
import urllib.request
import os

url = 'https://storage.googleapis.com/3d-shapes/3dshapes.h5'
path = '3dshapes.h5'

if not os.path.exists(path):
    print('Downloading 3D Shapes (~255MB)...')
    urllib.request.urlretrieve(url, path)
    print('Done.')
else:
    print('3D Shapes already downloaded.')

3D Shapes already downloaded.


In [ ]:
import h5py
import numpy as np

with h5py.File('3dshapes.h5', 'r') as f:
    imgs = f['images'][:]        # (480000, 64, 64, 3) uint8
    labels = f['labels'][:]      # (480000, 6) float64

print('Images shape:', imgs.shape)
print('Labels shape:', labels.shape)
print('Latent factors: floor hue, wall hue, object hue, scale, shape, orientation')

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

class Shapes3DDataset(Dataset):
    def __init__(self, images):
        self.images = images
        self.transform = T.Compose([
            T.ToTensor(),  # (H, W, C) uint8 -> (C, H, W) float [0,1]
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx])

dataset = Shapes3DDataset(imgs)
loader = DataLoader(dataset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
print(f'Dataset size: {len(dataset)}  |  Batches per epoch: {len(loader)}')

### Load the trained model
Same `VAE` architecture as `setuo.ipynb` (must match exactly to load the saved `state_dict`), then load `vae_trained.pt` instead of training from scratch.

In [ ]:
import torch.nn as nn

class VAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),   # 32x32
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, 2, 1),  # 16x16
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, 2, 1), # 8x8
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, 2, 1),# 4x4
            nn.ReLU(),
            nn.Flatten(),                # 256*4*4 = 4096
        )
        self.fc_mu     = nn.Linear(4096, latent_dim)
        self.fc_logvar = nn.Linear(4096, latent_dim)

        self.fc_decode = nn.Linear(latent_dim, 4096)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (256, 4, 4)),
            nn.ConvTranspose2d(256, 128, 4, 2, 1), # 8x8
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, 2, 1),  # 16x16
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, 2, 1),   # 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, 2, 1),    # 64x64
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(self.fc_decode(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

checkpoint = torch.load('vae_trained.pt', map_location=device)
model = VAE(latent_dim=checkpoint['latent_dim']).to(device)
model.load_state_dict(checkpoint['model_state'])
model.eval()
print(f"Loaded weights from vae_trained.pt (latent_dim={checkpoint['latent_dim']})")
print(model)

### Reconstruction quality check
Originals (top row) vs. reconstructions (bottom row) for a batch of images, using the loaded weights.

In [ ]:
import matplotlib.pyplot as plt

model.eval()
with torch.no_grad():
    sample_batch = next(iter(loader))[:8].to(device)
    recon, _, _ = model(sample_batch)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(sample_batch[i].cpu().permute(1, 2, 0))
    axes[0, i].axis('off')
    axes[1, i].imshow(recon[i].cpu().permute(1, 2, 0))
    axes[1, i].axis('off')

axes[0, 0].set_title('Original', loc='left')
axes[1, 0].set_title('Reconstructed', loc='left')
plt.tight_layout()
plt.show()

### Mutual Information Gap (MIG)
Copied verbatim from the last cell of `setuo.ipynb`.

In [ ]:
def mutual_information_gap(model, dataset, labels, device, n_samples=10000, n_bins=20):
    """
    Compute the Mutual Information Gap (MIG) disentanglement score.

    For each ground truth factor k:
      MIG_k = (MI(z_top1, v_k) - MI(z_top2, v_k)) / H(v_k)
    where z_top1 and z_top2 are the latent dims with the highest and
    second-highest MI with factor k.

    Overall MIG = mean over all factors. Range: [0, 1], higher is better.
    """
    model.eval()

    # Encode a random subset of images
    indices = np.random.choice(len(dataset), size=n_samples, replace=False)
    mus = []
    with torch.no_grad():
        for i in range(0, n_samples, 256):
            batch_idx = indices[i:i + 256]
            batch = torch.stack([dataset[int(j)] for j in batch_idx]).to(device)
            mu, _ = model.encode(batch)
            mus.append(mu.cpu().numpy())

    mus = np.concatenate(mus, axis=0)       # (n_samples, latent_dim)
    factors = labels[indices]               # (n_samples, 6)

    n_factors = factors.shape[1]
    latent_dim = mus.shape[1]

    # Bin continuous latents into n_bins discrete bins
    mus_binned = np.zeros_like(mus, dtype=int)
    for j in range(latent_dim):
        edges = np.linspace(mus[:, j].min(), mus[:, j].max(), n_bins + 1)[1:-1]
        mus_binned[:, j] = np.digitize(mus[:, j], bins=edges)

    # Map factor values to integer indices
    factors_binned = np.zeros_like(factors, dtype=int)
    for k in range(n_factors):
        unique_vals = np.unique(factors[:, k])
        val_to_idx = {v: i for i, v in enumerate(unique_vals)}
        factors_binned[:, k] = np.array([val_to_idx[v] for v in factors[:, k]])

    def entropy(x):
        _, counts = np.unique(x, return_counts=True)
        p = counts / counts.sum()
        return -np.sum(p * np.log(p + 1e-10))

    def mutual_information(z_j, v_k):
        h_vk = entropy(v_k)
        # H(V_k | Z_j) = sum_z p(z) * H(V_k | Z_j = z)
        h_vk_given_zj = sum(
            (z_j == z).mean() * entropy(v_k[z_j == z])
            for z in np.unique(z_j)
        )
        return h_vk - h_vk_given_zj

    # MI matrix: rows = latent dims, cols = factors
    mi_matrix = np.zeros((latent_dim, n_factors))
    for j in range(latent_dim):
        for k in range(n_factors):
            mi_matrix[j, k] = mutual_information(mus_binned[:, j], factors_binned[:, k])

    factor_names = ['floor hue', 'wall hue', 'object hue', 'scale', 'shape', 'orientation']
    mig_per_factor = []
    for k in range(n_factors):
        h_vk = entropy(factors_binned[:, k])
        sorted_mi = np.sort(mi_matrix[:, k])[::-1]
        mig_per_factor.append((sorted_mi[0] - sorted_mi[1]) / (h_vk + 1e-10))

    return np.mean(mig_per_factor), mig_per_factor, factor_names, mi_matrix

### Run the check
Calls the function above on the loaded model and prints the overall MIG plus the per-factor breakdown.

In [ ]:
mig, mig_per_factor, factor_names, mi_matrix = mutual_information_gap(model, dataset, labels, device)
print(f'Overall MIG: {mig:.4f}\n')
for name, score in zip(factor_names, mig_per_factor):
    print(f'  {name:15s}: {score:.4f}')